In [1]:
%pip install highspy highspy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Импорты
from pathlib import Path
import numpy as np
import highspy

##### Идея решения:

1. Находим множества, в которых вершины не соеденены между собой
2. Составляем задачу линейного программирования и решаем методом ветвей и границ
3. Вызможно 4 исхода:
   1. Решение пустое — выходим из узла
   2. Решение меньше текущего оптимального — выходим из узла
   3. Решение целочисленное — проверяем, является ли оно кликой и либо обновляем оптимум, либо добавляем доп. ограничение
   4. Решение дробное — выбираем дробную переменную ближе к 0.5 (как наиболее неопределённое) и ветвимся по нему

In [34]:
def read_graph(filepath) -> list[set]:
    filepath = Path(filepath)
    n = None
    edges = []

    with filepath.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("c"):
                continue
            parts = line.split()
            if parts[0] == "p":
                n = int(parts[2])
            elif parts[0] == "e":
                u = int(parts[1]) - 1
                v = int(parts[2]) - 1
                edges.append((u, v))

    graph = [set() for _ in range(n)]
    for u, v in edges:
        if u == v:
            continue
        graph[u].add(v)
        graph[v].add(u)

    return list(filter(None, graph))

def is_clique(graph: list[set]):
    n = len(graph)
    verts = list(graph.keys())
    for i in range(n):
        u = verts[i]
        for j in range(i + 1, n):
            v = verts[j]
            if v not in graph[u]:
                return False
    return True

def get_independent_sets(
    order: list, indep_sets: list, graph: list[set], merged_sets: list[set]
):
    if not order:
        return indep_sets
    start_v = order[0]
    indep_set = set()
    left_order = []
    for v in order:
        if v not in merged_sets[start_v]:
            indep_set.add(v)
            merged_sets[start_v] = merged_sets[start_v] | graph[v]
        else:
            left_order.append(v)
    indep_sets.append(indep_set)
    return get_independent_sets(left_order, indep_sets, graph, merged_sets)

def run(filepath):
    graph = read_graph(filepath)
    merged_sets = [s.copy() for s in graph] # С какими ребрами связаны вершины из одного независимого множества
    n = len(graph)
    order = sorted(range(n), key=lambda i: len(graph[i])) # Порядка мержа вершин (от самой одинокой к самой связанной)
    indep_sets = get_independent_sets(order, [], graph, merged_sets)


    print(indep_sets)

In [35]:
np.set_printoptions(precision=4, suppress=True)

print("-----------------------------")
inputs = Path("inputs/lab02")
for file in inputs.iterdir():
    run(file)

-----------------------------
[{0, 3, 4}, {1}, {2}]


In [ ]:
from __future__ import annotations

import math
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Set, Tuple, Optional

import highspy


EPS = 1e-7


def build_nonneighbors(adj: list[set[int]]) -> list[set[int]]:
    n = len(adj)
    all_vertices = set(range(n))
    nonneighbors = []
    for v in range(n):
        # все, кроме самой вершины и её соседей
        nn = all_vertices - adj[v] - {v}
        nonneighbors.append(nn)
    return nonneighbors


def find_nonedge_in_set(
    vertices: list[int], adj: list[set[int]]
) -> Optional[tuple[int, int]]:
    """Возвращает первую найденную пару несмежных вершин внутри множества."""
    k = len(vertices)
    for i in range(k):
        u = vertices[i]
        for j in range(i + 1, k):
            v = vertices[j]
            if v not in adj[u]:
                return (u, v)
    return None


def greedy_independent_set_from_pair(
    u: int,
    v: int,
    nonneighbors: list[set[int]],
    order: list[int],
) -> tuple[int, ...]:
    """
    Строим жадно независимое множество, начиная с пары (u, v),
    добавляя вершины, не смежные ни с одной уже выбранной.
    """
    indep = [u, v]
    current = set(indep)

    for w in order:
        if w in current:
            continue
        ok = True
        for z in indep:
            if w not in nonneighbors[z]:
                ok = False
                break
        if ok:
            indep.append(w)
            current.add(w)

    indep.sort()
    return tuple(indep)


def generate_initial_independent_sets(adj: list[set[int]]) -> list[tuple[int, ...]]:
    """
    Генерируем независимые множества примерно так, как просили в условии:
    перебираем пары несмежных вершин; если пара ещё не была покрыта ранее
    каким-то независимым множеством, строим из неё жадно максимальное
    независимое множество и добавляем ограничение sum x_i <= 1.

    Это НЕ исчерпывающий список всех независимых множеств, но модель заметно усиливает.
    """
    n = len(adj)
    nonneighbors = build_nonneighbors(adj)

    # порядок: вершины с большим числом несоседей раньше
    order = sorted(range(n), key=lambda x: len(nonneighbors[x]), reverse=True)

    covered_pairs = set()
    indep_sets = []
    seen_sets = set()

    for u in range(n):
        for v in range(u + 1, n):
            if v in adj[u]:
                continue  # есть ребро, пара нам не нужна
            pair = (u, v)
            if pair in covered_pairs:
                continue

            s = greedy_independent_set_from_pair(u, v, nonneighbors, order)
            if len(s) >= 2 and s not in seen_sets:
                indep_sets.append(s)
                seen_sets.add(s)

                # отмечаем все несмежные пары внутри найденного независимого множества
                for i in range(len(s)):
                    for j in range(i + 1, len(s)):
                        covered_pairs.add((s[i], s[j]))

    return indep_sets


def greedy_violated_independent_set(
    x: list[float],
    adj: list[set[int]],
    existing_cuts: set[tuple[int, ...]],
) -> Optional[tuple[int, ...]]:
    """
    Пытаемся найти нарушенное неравенство независимого множества:
        sum_{i in S} x_i <= 1

    Строим жадно независимое множество из вершин с наибольшими x_i.
    Если сумма > 1 + EPS, добавляем cut.
    """
    n = len(adj)
    nonneighbors = build_nonneighbors(adj)

    order = sorted(range(n), key=lambda i: x[i], reverse=True)

    indep = []
    for v in order:
        if x[v] <= EPS:
            continue
        ok = True
        for u in indep:
            if v not in nonneighbors[u]:
                ok = False
                break
        if ok:
            indep.append(v)

    indep.sort()
    cut = tuple(indep)
    if (
        len(cut) >= 2
        and sum(x[i] for i in cut) > 1.0 + 1e-6
        and cut not in existing_cuts
    ):
        return cut

    # запасной вариант: несколько запусков от хороших стартовых вершин
    top = [i for i in order if x[i] > EPS][:20]
    for start in top:
        indep = [start]
        for v in order:
            if v == start or x[v] <= EPS:
                continue
            ok = True
            for u in indep:
                if v not in nonneighbors[u]:
                    ok = False
                    break
            if ok:
                indep.append(v)

        indep.sort()
        cut = tuple(indep)
        if (
            len(cut) >= 2
            and sum(x[i] for i in cut) > 1.0 + 1e-6
            and cut not in existing_cuts
        ):
            return cut

    return None


@dataclass
class NodeResult:
    feasible: bool
    objective: float = 0.0
    x: list[float] = field(default_factory=list)


class MaxCliqueBnB:
    def __init__(self, adj: list[set[int]], time_limit_sec: float = 60.0):
        self.adj = adj
        self.n = len(adj)
        self.time_limit_sec = time_limit_sec
        self.start_time = time.time()

        self.initial_indep_sets = generate_initial_independent_sets(adj)

        self.best_clique: list[int] = []
        self.best_value = 0

        self.nodes_visited = 0
        self.lp_solved = 0
        self.cut_added = 0
        self.timed_out = False
        self.optimal_proved = False

    def time_exceeded(self) -> bool:
        return time.time() - self.start_time >= self.time_limit_sec

    def solve_lp(
        self,
        fixed_zero: set[int],
        fixed_one: set[int],
        extra_cuts: list[tuple[int, ...]],
    ) -> NodeResult:
        """
        Решает LP-релаксацию:
            max sum x_i
            s.t. sum_{i in S} x_i <= 1   для всех независимых множеств S
                 x_i in [0,1], либо фиксированы 0/1
        """
        highs = highspy.Highs()
        highs.setMaximize()

        inf = highspy.kHighsInf

        # Переменные
        for i in range(self.n):
            if i in fixed_zero:
                lb, ub = 0.0, 0.0
            elif i in fixed_one:
                lb, ub = 1.0, 1.0
            else:
                lb, ub = 0.0, 1.0
            highs.addVar(lb, ub)
            highs.changeColCost(i, 1.0)

        # Базовые ограничения по независимым множествам
        for s in self.initial_indep_sets:
            indices = list(s)
            values = [1.0] * len(indices)
            highs.addRow(-inf, 1.0, len(indices), indices, values)

        # Дополнительные cuts
        for s in extra_cuts:
            indices = list(s)
            values = [1.0] * len(indices)
            highs.addRow(-inf, 1.0, len(indices), indices, values)

        highs.run()
        self.lp_solved += 1

        status = highs.getModelStatus()

        if status in (
            highspy.HighsModelStatus.kInfeasible,
            highspy.HighsModelStatus.kUnboundedOrInfeasible,
            highspy.HighsModelStatus.kModelEmpty,
        ):
            return NodeResult(feasible=False)

        # Ожидаем оптимум LP
        if status != highspy.HighsModelStatus.kOptimal:
            # На всякий случай считаем, что этот узел нормально не решён.
            # Корректнее его не использовать для отсечения.
            return NodeResult(feasible=False)

        sol = highs.getSolution()
        info = highs.getInfo()

        x = list(sol.col_value)
        obj = info.objective_function_value
        return NodeResult(feasible=True, objective=obj, x=x)

    def pick_branch_var(
        self, x: list[float], fixed_zero: set[int], fixed_one: set[int]
    ) -> Optional[int]:
        """
        Выбираем дробную переменную, лучше всего ближайшую к 0.5
        """
        best_i = None
        best_score = None

        for i, val in enumerate(x):
            if i in fixed_zero or i in fixed_one:
                continue
            if EPS < val < 1.0 - EPS:
                score = abs(val - 0.5)
                if best_score is None or score < best_score:
                    best_score = score
                    best_i = i

        return best_i

    def current_selected_vertices(self, x: list[float]) -> list[int]:
        return [i for i, val in enumerate(x) if val >= 1.0 - 1e-6]

    def try_update_incumbent(self, clique_vertices: list[int]) -> None:
        if is_clique(clique_vertices, self.adj):
            if len(clique_vertices) > self.best_value:
                self.best_value = len(clique_vertices)
                self.best_clique = clique_vertices[:]

    def branch_and_bound(
        self,
        fixed_zero: set[int],
        fixed_one: set[int],
        inherited_cuts: list[tuple[int, ...]],
    ) -> None:
        if self.time_exceeded():
            self.timed_out = True
            return

        self.nodes_visited += 1

        # Невозможная фиксация
        if fixed_zero & fixed_one:
            return

        # Сразу отсекаем по очевидной границе: сколько максимум ещё можно добрать
        if (
            len(fixed_one) + (self.n - len(fixed_zero) - len(fixed_one))
            <= self.best_value
        ):
            return

        local_cuts = list(inherited_cuts)
        cut_set = set(local_cuts)

        while True:
            if self.time_exceeded():
                self.timed_out = True
                return

            res = self.solve_lp(fixed_zero, fixed_one, local_cuts)

            if not res.feasible:
                return

            # Отсечение по LP-границе
            if res.objective <= self.best_value + 1e-7:
                return

            x = res.x

            # Попробуем сгенерировать нарушенное ограничение независимого множества
            cut = greedy_violated_independent_set(x, self.adj, cut_set)
            if cut is not None:
                local_cuts.append(cut)
                cut_set.add(cut)
                self.cut_added += 1
                continue

            # Если решение целочисленное
            fractional = False
            for val in x:
                if EPS < val < 1.0 - EPS:
                    fractional = True
                    break

            if not fractional:
                chosen = [i for i, val in enumerate(x) if val >= 1.0 - 1e-6]

                # Честный чекер клики
                if is_clique(chosen, self.adj):
                    self.try_update_incumbent(chosen)
                    return

                # Если вдруг целочисленное решение не клика, добавим нарушенное ограничение
                bad_pair = find_nonedge_in_set(chosen, self.adj)
                if bad_pair is None:
                    # На всякий случай; сюда попасть не должны
                    return

                u, v = bad_pair
                # Построим из этой пары более сильный cut как независимое множество
                nonneighbors = build_nonneighbors(self.adj)
                order = sorted(range(self.n), key=lambda i: x[i], reverse=True)
                cut = greedy_independent_set_from_pair(u, v, nonneighbors, order)

                if cut not in cut_set:
                    local_cuts.append(cut)
                    cut_set.add(cut)
                    self.cut_added += 1
                    continue

                # Если такой cut уже есть, хотя решение всё ещё плохое, добавим просто пару
                pair_cut = tuple(sorted((u, v)))
                if pair_cut not in cut_set:
                    local_cuts.append(pair_cut)
                    cut_set.add(pair_cut)
                    self.cut_added += 1
                    continue

                return

            # Дробное решение -> ветвимся
            branch_var = self.pick_branch_var(x, fixed_zero, fixed_one)
            if branch_var is None:
                # Сюда в норме не попадём
                return

            # Полезная эвристика: сначала ветвь x_i = 1
            fixed_one_left = set(fixed_one)
            fixed_one_left.add(branch_var)

            fixed_zero_right = set(fixed_zero)
            fixed_zero_right.add(branch_var)

            self.branch_and_bound(fixed_zero, fixed_one_left, local_cuts)
            if self.time_exceeded():
                self.timed_out = True
                return

            self.branch_and_bound(fixed_zero_right, fixed_one, local_cuts)
            return

    def solve(self) -> dict:
        self.branch_and_bound(set(), set(), [])

        self.optimal_proved = not self.timed_out

        # Финальная честная проверка
        assert is_clique(self.best_clique, self.adj), (
            "Internal error: stored solution is not a clique"
        )

        return {
            "clique_size": self.best_value,
            "clique_vertices_1based": [v + 1 for v in self.best_clique],
            "nodes_visited": self.nodes_visited,
            "lp_solved": self.lp_solved,
            "cuts_added": self.cut_added,
            "timed_out": self.timed_out,
            "optimality_proved": self.optimal_proved,
            "elapsed_sec": time.time() - self.start_time,
        }


def solve_file(filepath: str | Path, time_limit_sec: float = 60.0) -> dict:
    n, adj = read_dimacs_clq(filepath)
    solver = MaxCliqueBnB(adj, time_limit_sec=time_limit_sec)
    result = solver.solve()

    print(f"File: {filepath}")
    print(f"Vertices: {n}")
    print(f"Best clique size: {result['clique_size']}")
    print(f"Clique (1-based): {result['clique_vertices_1based']}")
    print(f"Nodes visited: {result['nodes_visited']}")
    print(f"LP solved: {result['lp_solved']}")
    print(f"Cuts added: {result['cuts_added']}")
    print(f"Elapsed: {result['elapsed_sec']:.3f} sec")
    print(f"Timed out: {result['timed_out']}")
    print(f"Optimality proved: {result['optimality_proved']}")
    print("-" * 60)

    return result


def solve_directory(input_dir: str | Path, time_limit_sec: float = 60.0) -> None:
    input_dir = Path(input_dir)
    files = sorted([p for p in input_dir.iterdir() if p.is_file()])
    for filepath in files:
        solve_file(filepath, time_limit_sec=time_limit_sec)


if __name__ == "__main__":
    # пример:
    # solve_file("inputs/C125.9.clq", time_limit_sec=60.0)
    solve_directory("inputs", time_limit_sec=60.0)
